# Results chapter: figures, tables and numbers

Builds every figure and table of the Results chapter in `final_report/results_section/` (`results.md`, `appendix.md`). Run all cells top to bottom. The last cell copies the figures the chapter uses into `results_section/figures/`.

- **Code.** All functions come from `soc_eval.py`: the shared estimator ports, plus the "results chapter" block at the end of the file.
- **Outputs.** Figures go to `final_report/figures/res_*.{png,pdf}`. Tables go to `final_report/tables/res_*.{csv,md}`.
- **Run time.** About 25–35 minutes. Most of it is the offset correction on the LiFePO4 controlled run (1.26 million median steps) and the four operational replays.

**Rules this notebook follows** (from `result_report_instruction.md`):
- Every method uses the same timestamps of the same run.
- No reference SoC exists for the operational runs. Differences between methods are *method disagreement*.
- On the controlled runs the comparison line is the *sensor-derived characterization SoC*. Coulomb counting is not scored against it, and in-sample agreement is not used to rank methods.
- Unrecorded intervals are never bridged: lines break there, and the interval is hatched or marked.
- The controlled rest results are read from the tables of `controlled_evaluation.ipynb`, with the frozen settings. Cell 4.4 re-checks the LiNMC ones.

In [ ]:
import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'final_report' / 'soc_eval.py').exists()) / 'final_report'))
import soc_eval as se

se.apply_style()
pd.set_option('display.precision', 3)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)
models = {b: se.load_model(c['model']) for b, c in se.BATTERIES.items()}
OPS = [r for r, v in se.RUNS.items() if v['kind'] == 'operational']
MAIN_OPS = ['LiNMC General Test', 'LiFePO4 General Test']    # full figures in the main text
CHECK_CONTROLLED = True     # re-run the LiNMC controlled methods (under a minute) and compare with the saved rest table
NUM = {}                    # numbers quoted in results.md, printed at the end
FIGS = {}                   # figure number -> file name, for the figure map and the copy step

## 0. Data intervals and method configuration (Tables R1 and R2)
Every run in date order, with its ADC, checked against the logged files: the logged time step and the spare channels that never change.
- **ADS1115 era.** LiNMC controlled run: a 32 ms step, with ch3–ch5 and ch7 written as constants when the readings were re-encoded to ADS8688 codes.
- **ADS8688 era.** About 0.87–0.94 ms, all channels live.
- No run spans the change.

In [2]:
CONTROLLED_NOTES = {
    'LiNMC controlled': 'Four recording sessions about a week apart (18 Mar, 25 Mar, 1 Apr, 8 Apr). Battery current logged on '
                        'ch0 and remapped to ch2 (remap_linmc.js); load sensor on the same path; MPPT not connected '
                        '(set to 0 A); alternate slot not connected. The 8 Apr session follows a week of rest and is left '
                        'out of the controlled-run scores.',
    'LiFePO4 controlled': 'Eight recording files on one day; MPPT not connected (set to 0 A); load sensor reversed and '
                          'mirrored by remap_lifepo4.js; BMS cut-off at about 2.6 h, data below 30 V excluded.',
}
W, G, SIG = {}, {}, {}
for run, r in se.RUNS.items():
    W[run] = se.load_any(run)
    G[run] = se.gap_table(run, W[run])
    SIG[run] = se.adc_signature(se.TEST_DATA / r['folder'])
    assert SIG[run]['ADC from signature'] == r['adc'], f'{run}: logged signature does not match the declared ADC'

In [3]:
# The slow part (about 25 minutes), done once and used by every section below:
# - offset correction alone on the controlled runs (sections 4.1 and 4.2);
# - every method on each operational run, from the OCV-table start and from the fixed INIT_GRID starts (4.2 to 4.6).
KB = {run: se.kcl_batch(W[run], [se.OFFSETS]) for run, r in se.RUNS.items() if r['kind'] == 'controlled'}
OP = {}
for run in OPS:
    r = se.RUNS[run]
    _, est, extra, soc0 = se.run_operational(run, models[r['battery']], W[run])
    OP[run] = dict(win=W[run], est=est, extra=extra, soc0=soc0, gaps=G[run], state=se.record_state(W[run], est),
                   rests=se.find_rests(W[run], se.MIN_REST_OPERATIONAL), rests1=se.find_rests(W[run], 60))
    print(f"{run}: start {100 * soc0:.2f} %, {len(OP[run]['rests'])} verified rests (2 min), {len(OP[run]['rests1'])} with 1 min")

General Discharge Test: start 96.57 %, 0 verified rests (2 min), 3 with 1 min


General Discharge Test 2: start 87.83 %, 1 verified rests (2 min), 1 with 1 min


LiFePO4 Battery Discharge Test: start 10.40 %, 4 verified rests (2 min), 4 with 1 min


LiNMC Battery Discharge Test: start 93.42 %, 0 verified rests (2 min), 2 with 1 min


In [4]:
def fmt_len(s):
    return f'{s / 86400:.1f} d' if s > 86400 else f'{s / 3600:.1f} h' if s > 3600 else f'{s / 60:.1f} min' if s >= 60 else f'{s:.0f} s'


rows = {}
for run, r in se.RUNS.items():
    g = G[run][G[run].length_s > se.GAP_MIN]
    gap_text = '; '.join(f"{len(x)} {kind}{'s' if len(x) > 1 else ''}, total {fmt_len(x.length_s.sum())}"
                         for kind, x in g.groupby('kind')) or 'none'
    rows[r['short']] = {
        'date': r['date'], 'kind': r['kind'], 'battery (slot)': f"{r['battery']} ({r['slot']})", 'ADC': r['adc'],
        'logged step (ms)': SIG[run]['logged step (ms)'], 'constant spare channels': SIG[run]['constant spare channels'],
        'median window (s)': 10 * SIG[run]['logged step (ms)'] / 1e3,
        'recorded (h)': W[run].t.iloc[-1] / 3600, 'sessions': W[run].session.nunique(),
        'unrecorded intervals > 5 s': gap_text,
        'notes and applied corrections': se.OPERATIONAL_RUNS[run]['note'] if r['kind'] == 'operational' else CONTROLLED_NOTES[run],
    }
r1 = pd.DataFrame(rows).T
display(Markdown(se.md_table(r1, 'data_intervals', floatfmt='.3g')))

|                       | date                | kind        | battery (slot)      | ADC     |   logged step (ms) | constant spare channels   |   median window (s) |   recorded (h) |   sessions | unrecorded intervals > 5 s       | notes and applied corrections                                                                                                                                                                                                                                                                                                      |
|:----------------------|:--------------------|:------------|:--------------------|:--------|-------------------:|:--------------------------|--------------------:|---------------:|-----------:|:---------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| LiNMC controlled      | 18 Mar – 8 Apr 2026 | controlled  | LiNMC (main)        | ADS1115 |             32     | ch3, ch4, ch5, ch7        |             0.32    |          8.68  |          5 | 4 recording breaks, total 20.6 d | Four recording sessions about a week apart (18 Mar, 25 Mar, 1 Apr, 8 Apr). Battery current logged on ch0 and remapped to ch2 (remap_linmc.js); load sensor on the same path; MPPT not connected (set to 0 A); alternate slot not connected. The 8 Apr session follows a week of rest and is left out of the controlled-run scores. |
| LiFePO4 controlled    | 25 May 2026         | controlled  | LiFePO4 (main)      | ADS8688 |              0.938 | none                      |             0.00938 |          3.43  |          8 | 7 recording breaks, total 1.7 h  | Eight recording files on one day; MPPT not connected (set to 0 A); load sensor reversed and mirrored by remap_lifepo4.js; BMS cut-off at about 2.6 h, data below 30 V excluded.                                                                                                                                                    |
| GDT 1 (LiNMC, 12 Jun) | 12 Jun 2026         | operational | LiNMC (main)        | ADS8688 |              0.866 | none                      |             0.00866 |          0.986 |          1 | none                             | 12 Jun, charging near full. The load sensor reads -0.22 A when off (learned by offset correction). The separate 1.4-min recording at 16:37 (79 s gap, 0.2 A trickle charge) is left out.                                                                                                                                           |
| GDT 2 (LiNMC, 12 Jun) | 12 Jun 2026         | operational | LiNMC (main)        | ADS8688 |              0.866 | none                      |             0.00866 |          0.991 |          1 | none                             | 12 Jun, charging near full.                                                                                                                                                                                                                                                                                                        |
| LiFePO4 (22 Jun)      | 22 Jun 2026         | operational | LiFePO4 (alternate) | ADS8688 |              0.866 | none                      |             0.00866 |          0.696 |          1 | 2 logger stalls, total 2.3 min   | 22 Jun. LiFePO4 in the alternate slot; the LiNMC in the main slot is idle and not evaluated. The server restarted during the test (run ids 1-8); the replay runs straight through.                                                                                                                                                 |
| LiNMC (29 Jun)        | 29 Jun 2026         | operational | LiNMC (main)        | ADS8688 |              0.866 | none                      |             0.00866 |          1.48  |          1 | 32 logger stalls, total 10.4 min | 29 Jun. MPPT flip latch disabled (one -10.24 A spike at 0.93 h would negate every later MPPT reading). 10.4 min of logger stalls, mostly at rest; 0.24 Ah passed unrecorded (at most 0.34 pp).                                                                                                                                     |

In [5]:
def ekf_text(b):
    e = se.BATTERIES[b]['ekf']
    text = (f"q_soc {e['q_soc']:g} /s, q_rc {e['q_rc']:g} V²/s, R = {e['r']:g} V²·s ÷ dt, P0_soc {e['p0_soc']:g}; "
            f"RC voltages {'relax over' if se.BATTERIES[b]['gap_decay'] else 'reset at'} recording breaks")
    if e.get('gate_v'):
        text += (f"; voltage update only where |dOCV/dSoC| ≥ {e['h_min']:g} V per unit SoC, or after {e['gate_t']:g} s with "
                 f"|I| < {e['gate_i']:g} A when the rested voltage maps to such a part of the curve")
    return text


o = se.OFFSETS
r2 = pd.DataFrame({
    'Coulomb counting': {
        'current input': 'offset-corrected battery current',
        'voltage input': 'none',
        'model parameters used': 'Q_total',
        'LiNMC settings': '–', 'LiFePO4 settings': '–',
        'initial state': 'SoC0 (no correction after the start)'},
    'OCV mapping': {
        'current input': 'none',
        'voltage input': 'terminal voltage at every step (short-rest curve applied with no IR compensation, also under current)',
        'model parameters used': 'inverse of the OCV table',
        'LiNMC settings': '–', 'LiFePO4 settings': '–',
        'initial state': 'none (no memory): not applicable'},
    'EKF': {
        'current input': 'offset-corrected battery current',
        'voltage input': 'terminal voltage (2RC model prediction)',
        'model parameters used': 'Q_total; OCV, R0, R1, C1, R2, C2 against SoC',
        'LiNMC settings': ekf_text('LiNMC'), 'LiFePO4 settings': ekf_text('LiFePO4'),
        'initial state': 'SoC0, RC voltages 0'},
    'EKF + KCL': {
        'current input': (f"battery current after offsets + gated KCL update (R {se.KCL_GATED['r']:g} A²·s ÷ dt, "
                          f"{se.KCL_GATED['gate']:g}σ innovation gate)"),
        'voltage input': 'as EKF',
        'model parameters used': 'as EKF',
        'LiNMC settings': 'as EKF', 'LiFePO4 settings': 'as EKF',
        'initial state': 'as EKF'},
}).T
SHARED = (f"All methods: 10-sample median stream (as kalman_filter.js), samples below the {se.V_CUTOFF} V cutoff excluded, "
          f"records every 1 s. Offset correction (shared input step, not a method): a sensor is idle when load or MPPT reads "
          f"< {o['idle_th']} A, or a battery reads < {o['idle_bat_th']} A while load and MPPT are idle, for {o['idle_t']:g} s; "
          f"its offset is learned while idle (R {o['r_idle']:g} A²·s ÷ dt, process noise {o['q_load']:g} A²/s) and subtracted "
          f"while active; readings below {o['th']} A are output as 0 A.")
display(Markdown(se.md_table(r2, 'method_config')))
print(SHARED)
NUM['shared settings'] = SHARED

|                  | current input                                                                             | voltage input                                                                                         | model parameters used                        | LiNMC settings                                                                                            | LiFePO4 settings                                                                                                                                                                                                                                              | initial state                        |
|:-----------------|:------------------------------------------------------------------------------------------|:------------------------------------------------------------------------------------------------------|:---------------------------------------------|:----------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------|
| Coulomb counting | offset-corrected battery current                                                          | none                                                                                                  | Q_total                                      | –                                                                                                         | –                                                                                                                                                                                                                                                             | SoC0 (no correction after the start) |
| OCV mapping      | none                                                                                      | terminal voltage at every step (short-rest curve applied with no IR compensation, also under current) | inverse of the OCV table                     | –                                                                                                         | –                                                                                                                                                                                                                                                             | none (no memory): not applicable     |
| EKF              | offset-corrected battery current                                                          | terminal voltage (2RC model prediction)                                                               | Q_total; OCV, R0, R1, C1, R2, C2 against SoC | q_soc 1e-12 /s, q_rc 1e-07 V²/s, R = 0.0001 V²·s ÷ dt, P0_soc 0.01; RC voltages reset at recording breaks | q_soc 1e-09 /s, q_rc 1e-07 V²/s, R = 0.0001 V²·s ÷ dt, P0_soc 0.01; RC voltages relax over recording breaks; voltage update only where |dOCV/dSoC| ≥ 5 V per unit SoC, or after 300 s with |I| < 1 A when the rested voltage maps to such a part of the curve | SoC0, RC voltages 0                  |
| EKF + KCL        | battery current after offsets + gated KCL update (R 0.0025 A²·s ÷ dt, 4σ innovation gate) | as EKF                                                                                                | as EKF                                       | as EKF                                                                                                    | as EKF                                                                                                                                                                                                                                                        | as EKF                               |

All methods: 10-sample median stream (as kalman_filter.js), samples below the 30 V cutoff excluded, records every 1 s. Offset correction (shared input step, not a method): a sensor is idle when load or MPPT reads < 0.3 A, or a battery reads < 1.0 A while load and MPPT are idle, for 10 s; its offset is learned while idle (R 0.001 A²·s ÷ dt, process noise 1e-07 A²/s) and subtracted while active; readings below 0.3 A are output as 0 A.


## 4.1 Battery characterization (Figures R1–R3, Table R3)
- **Run figures.** Battery current and terminal voltage over each controlled discharge, plotted on recorded time with the unrecorded gaps removed. Lines break at every gap, and each gap's length is printed.
- **Short-rest voltage figure.** The points are the knots of the OCV curve, exactly as `3.ocv-soc-fit.ipynb` printed them when it built the model; this cell checks them against the model's OCV table. The curve is a PCHIP interpolant, so its residual at the knots is 0 by construction. The leave-one-out and log-model residuals are descriptive, on the same points.

In [6]:
CHAR, KNOTS, POINTS, FITS, FLAGS, CRESTS = {}, {}, {}, {}, {}, {}
for b in ('LiNMC', 'LiFePO4'):
    run = f'{b} controlled'
    CRESTS[b] = se.find_rests(W[run])
    k, p = se.ocv_fit_residuals(se.ocv_knots(b))
    ok = k.soc.between(0, 100)
    assert np.abs(se.soc_to_ocv(k.soc[ok] / 100, models[b]) - k.V[ok]).max() < 2e-3, f'{b}: knots do not match the model'
    CHAR[b], POINTS[b] = se.characterization_table(W[run], CRESTS[b], b, models[b], k)
    KNOTS[b], FITS[b] = k.assign(battery=b), (k, p)
    FLAGS[b] = POINTS[b].ref_soc_end_pct[~POINTS[b].criterion_met].to_numpy()
    fig = se.plot_char_run(W[run], CRESTS[b], G[run], f'{b} controlled discharge')
    se.save(fig, f'res_char_run_{b}')
    plt.show()
FIGS['R1'], FIGS['R2'] = 'res_char_run_LiNMC', 'res_char_run_LiFePO4'
char = pd.concat(CHAR.values(), axis=1)
display(Markdown(se.md_table(char, 'char', floatfmt='.2f')))
knots = pd.concat(KNOTS.values(), ignore_index=True)[['battery', 'soc', 'V', 'kind', 'loo_mV', 'log_mV']]
display(Markdown(se.md_table(knots, 'char_knots', floatfmt='.2f', index=False)))
points = pd.concat({b: p for b, p in POINTS.items()}, names=['battery', 'rest'])
se.md_table(points.drop(columns=[c for c in points if c.startswith('ref_soc_t0')]), 'char_rest_points', floatfmt='.2f')
fig = se.plot_rest_voltage_fit(FITS, FLAGS)
se.save(fig, 'res_rest_voltage_fit')
FIGS['R3'] = 'res_rest_voltage_fit'
plt.show()
for b in CHAR:
    m = KNOTS[b][KNOTS[b].measured]
    NUM[f'{b}: largest |log-model residual|'] = m.loc[m.log_mV.abs().idxmax(), ['soc', 'V', 'log_mV']].astype(float).round(2).to_dict()
    NUM[f'{b}: largest |leave-one-out residual|'] = m.loc[m.loo_mV.abs().idxmax(), ['soc', 'V', 'loo_mV']].astype(float).round(2).to_dict()
    NUM[f'{b}: rests where 10 mV/min was not reached (SoC %)'] = np.round(FLAGS[b], 1).tolist()

C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\627575608.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\627575608.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


|                                                     | LiNMC                                                  | LiFePO4                                                        |
|:----------------------------------------------------|:-------------------------------------------------------|:---------------------------------------------------------------|
| ADC                                                 | ADS1115                                                | ADS8688                                                        |
| recorded                                            | 18 Mar – 8 Apr 2026; 5 sessions, 8.68 h logged         | 25 May 2026; 8 sessions, 3.43 h logged                         |
| start of discharge                                  | first sample, 18 Mar 16:00, 58.20 V                    | first sample, 25 May 11:54, 52.42 V                            |
| end of discharge                                    | last sample above 30 V before the BMS cut-off, 43.00 V | last sample above 30 V before the BMS cut-off, 47.32 V         |
| charge out, battery sensor (Ah)                     | 69.84                                                  | 37.00                                                          |
| charge out, load sensor (Ah)                        | 72.34                                                  | 37.05                                                          |
| charge out, mean of the two raw sensors (Ah)        | 71.09                                                  | 37.03                                                          |
| usable capacity Q_total (Ah)                        | 71.43                                                  | 37.01                                                          |
| Q_total as % of rated capacity                      | 71.43                                                  | 74.02                                                          |
| discharge steps (battery |I| > 1 A, 30 s or more)   | 14                                                     | 11                                                             |
| step mean current, min – max (A)                    | 26.2 – 31.1                                            | 25.5 – 30.4                                                    |
| step length, min / median / max (min)               | 4.9 / 11.1 / 12.6                                      | 0.7 / 9.5 / 10.4                                               |
| rests (all currents < 0.5 A, 5 min or more)         | 13                                                     | 6                                                              |
| rest length, min / median / max (min)               | 16.8 / 31.3 / 36.3                                     | 7.1 / 10.2 / 13.0                                              |
| rest-end voltages in the OCV curve (measured knots) | 13                                                     | 7                                                              |
| constructed knots                                   | end estimate (last loaded sample + I·R)                | end estimate (last loaded sample + I·R), extrapolated to 100 % |
| logged voltage reached 10 mV/min (rests)            | 13 of 13                                               | 4 of 6                                                         |
| t₁ − t₀ where reached, min – max (min)              | 5.4 – 15.6                                             | 2.9 – 9.6                                                      |
| voltage change after t₁, max (mV)                   | 125.86                                                 | 24.31                                                          |
| time below the cutoff, excluded (h)                 | 0.07                                                   | 0.82                                                           |
| leave-one-out residual, RMS / max |·| (mV)          | 246 / 728                                              | 145 / 345                                                      |
| log-model residual, RMS / max |·| (mV)              | 131 / 302                                              | 40 / 63                                                        |

| battery   |    soc |     V | kind                                    |   loo_mV |   log_mV |
|:----------|-------:|------:|:----------------------------------------|---------:|---------:|
| LiNMC     |  -0.11 | 44.90 | end estimate (last loaded sample + I·R) |   nan    |   nan    |
| LiNMC     |   3.28 | 46.73 | rest end                                |  -128.76 |   -10.63 |
| LiNMC     |  10.17 | 49.10 | rest end                                |   727.73 |   143.52 |
| LiNMC     |  17.22 | 49.44 | rest end                                |  -349.65 |  -302.22 |
| LiNMC     |  24.24 | 50.26 | rest end                                |   173.84 |    31.78 |
| LiNMC     |  31.88 | 50.84 | rest end                                |    -5.60 |   153.69 |
| LiNMC     |  39.34 | 51.33 | rest end                                |    23.25 |   140.45 |
| LiNMC     |  47.23 | 51.80 | rest end                                |   -45.70 |   -13.87 |
| LiNMC     |  54.22 | 52.38 | rest end                                |   -14.66 |   -99.04 |
| LiNMC     |  61.16 | 53.12 | rest end                                |    -1.52 |  -120.84 |
| LiNMC     |  68.64 | 54.10 | rest end                                |  -142.67 |   -72.92 |
| LiNMC     |  76.31 | 55.41 | rest end                                |   203.33 |   191.65 |
| LiNMC     |  84.81 | 56.33 | rest end                                |    37.30 |   -28.72 |
| LiNMC     |  92.19 | 57.05 | rest end                                |  -157.73 |   -12.84 |
| LiNMC     | 100.00 | 58.20 | rested start                            |   nan    |   nan    |
| LiFePO4   |  -0.22 | 49.12 | end estimate (last loaded sample + I·R) |   nan    |   nan    |
| LiFePO4   |  10.96 | 51.24 | rest end                                |   344.67 |     7.54 |
| LiFePO4   |  22.82 | 51.90 | rest end                                |    -5.73 |   -41.66 |
| LiFePO4   |  34.83 | 52.19 | rest end                                |    87.02 |    63.15 |
| LiFePO4   |  46.86 | 52.22 | rest end                                |   -33.58 |    -6.46 |
| LiFePO4   |  59.29 | 52.33 | rest end                                |   -83.14 |   -59.12 |
| LiFePO4   |  70.49 | 52.63 | rest end                                |   101.58 |    43.94 |
| LiFePO4   |  83.95 | 52.83 | rest end                                |   -54.43 |    -7.40 |
| LiFePO4   | 100.00 | 53.06 | extrapolated to 100 %                   |   nan    |   nan    |

C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\627575608.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.2 Current sensor offset (Figure R4, Tables R4–R6)
- **Zero-current windows.** These use the idle rule that offset correction implements. Load or MPPT is idle below 0.3 A. A battery is idle below 1 A while load and MPPT are idle and its voltage is above the cutoff. A window starts 10 s after the rule begins to hold, which also skips the switching transient, and it must then last at least 20 s.
- **Channels with no sensor are left out.** On the controlled runs the MPPT channel is a constant written by the remap scripts, and the alternate slot is not connected.
- **The "after" value.** After correction the output is 0 A by the 0.3 A floor, so that number is a consequence of the rule, not a measurement. The table therefore also reports the reading minus the learned offset, before the floor.
- No temperature channel was logged, and there was no independent current reference during the runs.

In [7]:
OFFW, BIAS, DUR = [], {}, {}
for run, r in se.RUNS.items():
    win = W[run]
    if r['kind'] == 'controlled':      # MPPT channel written as 0 A by the remap scripts; alternate slot not connected
        chans, t_rec, bias = ['I_b1', 'I_load'], KB[run]['t_rec'], KB[run]['bias'][0]
    else:
        chans, t_rec, bias = ['I_b1', 'I_b2', 'I_load', 'I_mppt'], OP[run]['est'].t.to_numpy(), OP[run]['extra']['bias_offsets']
    w = se.learned_offsets(win, se.idle_windows(win, chans), t_rec, bias)
    rec = np.searchsorted(win.t.to_numpy(), t_rec)
    conn = {c: (win[f'V_b{c[-1]}'].to_numpy()[rec] > se.V_CUTOFF) if c in ('I_b1', 'I_b2') else np.ones(len(rec), bool)
            for c in chans}
    BIAS[run] = (t_rec, bias, {c: m for c, m in conn.items() if m.any()})
    DUR[run] = win.t.iloc[-1] / 3600
    OFFW.append(w.assign(run=r['short'], adc=r['adc']))
OFFW = pd.concat(OFFW, ignore_index=True)
OFFW.drop(columns='idx').to_csv(se.TABLE_DIR / 'res_offset_windows.csv', index=False)
offsets = se.offset_table(OFFW)
display(Markdown(se.md_table(offsets, 'offsets')))
fig = se.plot_offset_timeline(OFFW.assign(run=OFFW.run.map({v['short']: k for k, v in se.RUNS.items()})), BIAS, DUR)
se.save(fig, 'res_offset_timeline')
FIGS['R4'] = 'res_offset_timeline'
plt.show()

|                                                 | ADC     |   windows |   idle time (min) |   offset: median reading (A) | lowest / highest window median (A)   | first → last window median (A)   |   noise: median within-window SD (A) |   after removing learned offset: median (A) |   after removing learned offset: max |·| (A) |
|:------------------------------------------------|:--------|----------:|------------------:|-----------------------------:|:-------------------------------------|:---------------------------------|-------------------------------------:|--------------------------------------------:|---------------------------------------------:|
| ('LiNMC controlled', 'Battery, main slot')      | ADS1115 |        15 |           361.622 |                       -0.163 | -0.192 / -0.079                      | -0.170 → -0.096                  |                                0.007 |                                       0.000 |                                        0.062 |
| ('LiNMC controlled', 'Load')                    | ADS1115 |        15 |           361.622 |                        0.034 | -0.053 / +0.067                      | -0.008 → +0.045                  |                                0.007 |                                      -0.000 |                                        0.014 |
| ('LiFePO4 controlled', 'Battery, main slot')    | ADS8688 |         9 |            72.824 |                       -0.039 | -0.088 / -0.007                      | -0.059 → -0.047                  |                                0.025 |                                       0.000 |                                        0.013 |
| ('LiFePO4 controlled', 'Load')                  | ADS8688 |        11 |           114.402 |                       -0.061 | -0.088 / -0.033                      | -0.088 → -0.033                  |                                0.024 |                                       0.000 |                                        0.016 |
| ('GDT 1 (LiNMC, 12 Jun)', 'Battery, main slot') | ADS8688 |         1 |             0.412 |                       -0.154 | -0.154 / -0.154                      | -0.154 → -0.154                  |                                0.021 |                                      -0.009 |                                        0.009 |
| ('GDT 1 (LiNMC, 12 Jun)', 'Load')               | ADS8688 |         4 |             1.541 |                       -0.218 | -0.224 / -0.213                      | -0.224 → -0.213                  |                                0.028 |                                      -0.000 |                                        0.003 |
| ('GDT 1 (LiNMC, 12 Jun)', 'MPPT output')        | ADS8688 |         3 |             6.741 |                        0.109 | +0.105 / +0.145                      | +0.109 → +0.145                  |                                0.024 |                                      -0.001 |                                        0.043 |
| ('GDT 2 (LiNMC, 12 Jun)', 'Battery, main slot') | ADS8688 |         2 |             1.774 |                       -0.158 | -0.159 / -0.158                      | -0.159 → -0.158                  |                                0.020 |                                      -0.004 |                                        0.006 |
| ('GDT 2 (LiNMC, 12 Jun)', 'Load')               | ADS8688 |        26 |            26.846 |                       -0.213 | -0.223 / -0.180                      | -0.180 → -0.208                  |                                0.025 |                                       0.000 |                                        0.015 |
| ('GDT 2 (LiNMC, 12 Jun)', 'MPPT output')        | ADS8688 |         1 |             2.431 |                        0.054 | +0.054 / +0.054                      | +0.054 → +0.054                  |                                0.026 |                                      -0.004 |                                        0.004 |
| ('LiFePO4 (22 Jun)', 'Battery, main slot')      | ADS8688 |         6 |            20.503 |                        0.001 | +0.001 / +0.001                      | +0.001 → +0.001                  |                                0.011 |                                      -0.000 |                                        0.000 |
| ('LiFePO4 (22 Jun)', 'Battery, alternate slot') | ADS8688 |         6 |            20.503 |                        0.002 | +0.001 / +0.002                      | +0.001 → +0.001                  |                                0.008 |                                       0.000 |                                        0.001 |
| ('LiFePO4 (22 Jun)', 'Load')                    | ADS8688 |         6 |            20.503 |                       -0.011 | -0.012 / -0.001                      | -0.001 → -0.011                  |                                0.023 |                                      -0.002 |                                        0.006 |
| ('LiFePO4 (22 Jun)', 'MPPT output')             | ADS8688 |         1 |            41.611 |                       -0.013 | -0.013 / -0.013                      | -0.013 → -0.013                  |                                0.019 |                                      -0.001 |                                        0.001 |
| ('LiNMC (29 Jun)', 'Battery, main slot')        | ADS8688 |         5 |            14.466 |                        0.002 | +0.001 / +0.534                      | +0.001 → +0.463                  |                                0.030 |                                      -0.000 |                                        0.177 |
| ('LiNMC (29 Jun)', 'Load')                      | ADS8688 |         4 |            53.910 |                       -0.003 | -0.016 / +0.014                      | +0.008 → +0.014                  |                                0.038 |                                      -0.003 |                                        0.006 |
| ('LiNMC (29 Jun)', 'MPPT output')               | ADS8688 |         5 |            32.281 |                        0.001 | +0.001 / +0.002                      | +0.001 → +0.001                  |                                0.014 |                                      -0.001 |                                        0.001 |

C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\4065007298.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Effect on the integrated charge of the evaluated battery, and the observed offset projected over PROJECTION_H
rows = {}
for run, r in se.RUNS.items():
    win, b = W[run], r['battery']
    Q = models[b]['Q_total']
    col = 'I_b1' if r['slot'] == 'main' else 'I_b2'
    if r['kind'] == 'controlled':
        valid, dt = win.V_bat.to_numpy() > se.V_CUTOFF, win.dt.to_numpy()
        ah = lambda x: np.sum(np.where(valid, x * dt, 0.0)) / 3600
        raw, corr = ah(win.I_bat.to_numpy()), ah(KB[run]['i_b1' if r['slot'] == 'main' else 'i_b2'][0])
    else:
        raw, corr = OP[run]['extra']['Ah']['raw'], OP[run]['extra']['Ah']['offsets']
    off = OFFW[(OFFW.run == r['short']) & (OFFW.col == col)].median_A.median()
    rows[r['short']] = {
        'battery': b, 'Q_total (Ah)': Q, 'recorded (h)': win.t.iloc[-1] / 3600,
        'net charge out, raw current (Ah)': raw, 'net charge out, offset-corrected (Ah)': corr,
        'corrected − raw (Ah)': corr - raw, 'corrected − raw (pp of Q_total)': 100 * (corr - raw) / Q,
        'battery sensor zero reading, median (A)': off,
        f'that offset over {se.PROJECTION_H:g} h (Ah)': abs(off) * se.PROJECTION_H,
        f'that offset over {se.PROJECTION_H:g} h (pp of Q_total)': 100 * abs(off) * se.PROJECTION_H / Q,
    }
charge = pd.DataFrame(rows).T
display(Markdown(se.md_table(charge, 'offset_charge')))

# The ADC change: zero readings of the battery and load sensors on either side (no run spans it)
bl = OFFW[OFFW.col.isin(['I_b1', 'I_b2', 'I_load'])].assign(sensor=lambda d: d.sensor.where(d.col == 'I_load', 'Battery (either slot)'))
g = bl.groupby(['adc', 'sensor'])
adc = pd.DataFrame({'runs': g.run.nunique(), 'windows': g.size(), 'median window median (A)': g.median_A.median(),
                    'lowest / highest window median (A)': g.median_A.agg(lambda x: f'{x.min():+.3f} / {x.max():+.3f}'),
                    'median within-window SD (A)': g.sd_A.median()})
display(Markdown(se.md_table(adc, 'offset_adc')))

|                       | battery   |   Q_total (Ah) |   recorded (h) |   net charge out, raw current (Ah) |   net charge out, offset-corrected (Ah) |   corrected − raw (Ah) |   corrected − raw (pp of Q_total) |   battery sensor zero reading, median (A) |   that offset over 12 h (Ah) |   that offset over 12 h (pp of Q_total) |
|:----------------------|:----------|---------------:|---------------:|-----------------------------------:|----------------------------------------:|-----------------------:|----------------------------------:|------------------------------------------:|-----------------------------:|----------------------------------------:|
| LiNMC controlled      | LiNMC     |         71.426 |          8.679 |                             69.843 |                                  71.193 |                  1.350 |                             1.890 |                                    -0.163 |                        1.950 |                                   2.730 |
| LiFePO4 controlled    | LiFePO4   |         37.012 |          3.432 |                             36.998 |                                  37.089 |                  0.091 |                             0.245 |                                    -0.039 |                        0.466 |                                   1.259 |
| GDT 1 (LiNMC, 12 Jun) | LiNMC     |         71.426 |          0.986 |                             -1.611 |                                  -1.512 |                  0.099 |                             0.138 |                                    -0.154 |                        1.851 |                                   2.592 |
| GDT 2 (LiNMC, 12 Jun) | LiNMC     |         71.426 |          0.991 |                             -1.841 |                                  -1.787 |                  0.054 |                             0.076 |                                    -0.158 |                        1.901 |                                   2.661 |
| LiFePO4 (22 Jun)      | LiFePO4   |         37.012 |          0.696 |                              0.585 |                                   0.584 |                 -0.001 |                            -0.003 |                                     0.002 |                        0.021 |                                   0.057 |
| LiNMC (29 Jun)        | LiNMC     |         71.426 |          1.481 |                             -2.853 |                                  -3.106 |                 -0.253 |                            -0.354 |                                     0.002 |                        0.028 |                                   0.040 |

|                                      |   runs |   windows |   median window median (A) | lowest / highest window median (A)   |   median within-window SD (A) |
|:-------------------------------------|-------:|----------:|---------------------------:|:-------------------------------------|------------------------------:|
| ('ADS1115', 'Battery (either slot)') |      1 |        15 |                     -0.163 | -0.192 / -0.079                      |                         0.007 |
| ('ADS1115', 'Load')                  |      1 |        15 |                      0.034 | -0.053 / +0.067                      |                         0.007 |
| ('ADS8688', 'Battery (either slot)') |      5 |        29 |                      0.001 | -0.159 / +0.534                      |                         0.020 |
| ('ADS8688', 'Load')                  |      5 |        51 |                     -0.211 | -0.224 / +0.014                      |                         0.025 |

## 4.3 Operational runs (Figures R5–R7, Tables R7–R9)
Every method starts from the OCV table at the first sample above the cutoff, as the server does on a fresh start: the table row whose OCV is nearest the voltage. The same call also replays CC, the EKF and EKF + KCL from the fixed starts of Section 4.5.
- **Time axis.** Wall-clock elapsed time. Logger stalls are hatched, and lines break there.
- **States.** These are inferred from the currents, because no switch-position channel was logged. A *verified rest* has every current below 0.5 A for at least 2 min, with the first 10 s left out.

In [9]:
rows, pair, bystate = {}, {}, {}
for run, o in OP.items():
    est, st, gaps = o['est'], o['state'], o['gaps']
    ok = np.flatnonzero(est.V > se.V_CUTOFF)
    f, l = ok[0], ok[-1]
    dt = pd.Series(np.r_[0.0, np.diff(est.t.to_numpy())])
    row = {'first-sample voltage (V)': o['win'].V_bat[o['win'].V_bat > se.V_CUTOFF].iloc[0],
           'start SoC, all methods (%)': 100 * o['soc0'],
           'battery current at the first record (A)': est.I_raw.iloc[f], 'state at the first record': st[f],
           'recorded (h)': est.t.iloc[l] / 3600, 'elapsed wall clock (h)': se.wall_hours([est.t.iloc[l]], gaps)[0]}
    row.update({f'{s} (min)': dt[st == s].sum() / 60 for s in se.STATE_LANES})
    row['verified rests (2 min)'] = len(o['rests'])
    row.update({f'{name}: end (%)': 100 * est[col].iloc[l] for col, name in se.METHODS.items()})
    rows[se.RUNS[run]['short']] = row
    pair[se.RUNS[run]['short']] = se.pairwise_summary(est, st, gaps)
    bystate[se.RUNS[run]['short']] = se.disagreement_by_state(est, st)
start_end = pd.DataFrame(rows).T
display(Markdown(se.md_table(start_end, 'op_start_end', floatfmt='.2f')))
pair = pd.concat(pair, names=['run', 'pair'])
display(Markdown(se.md_table(pair, 'op_pairwise')))
bystate = pd.concat(bystate, names=['run', 'state'])
display(Markdown(se.md_table(bystate, 'op_by_state')))

|                       |   first-sample voltage (V) |   start SoC, all methods (%) |   battery current at the first record (A) | state at the first record   |   recorded (h) |   elapsed wall clock (h) |   charging (min) |   discharging (min) |   pass-through (min) |   transition (min) |   rest (min) |   verified rests (2 min) |   Coulomb counting: end (%) |   OCV mapping: end (%) |   EKF: end (%) |   EKF + KCL: end (%) |
|:----------------------|---------------------------:|-----------------------------:|------------------------------------------:|:----------------------------|---------------:|-------------------------:|-----------------:|--------------------:|---------------------:|-------------------:|-------------:|-------------------------:|----------------------------:|-----------------------:|---------------:|---------------------:|
| GDT 1 (LiNMC, 12 Jun) |                      57.64 |                        96.57 |                                     -0.19 | rest                        |           0.99 |                     0.99 |            50.60 |                2.82 |                 0.00 |               0.00 |         5.73 |                        0 |                       98.68 |                  98.89 |          98.77 |                98.75 |
| GDT 2 (LiNMC, 12 Jun) |                      56.62 |                        87.83 |                                     -0.24 | rest                        |           0.99 |                     0.99 |            55.57 |                0.00 |                 0.00 |               0.00 |         3.88 |                        1 |                       90.33 |                  92.08 |          91.14 |                91.12 |
| LiFePO4 (22 Jun)      |                      51.19 |                        10.40 |                                     -0.01 | rest                        |           0.70 |                     0.73 |             0.00 |               18.55 |                 0.00 |               0.00 |        23.22 |                        4 |                        8.83 |                   3.54 |           3.55 |                 3.55 |
| LiNMC (29 Jun)        |                      57.20 |                        93.42 |                                      0.01 | rest                        |           1.48 |                     1.65 |            53.48 |               21.52 |                 0.18 |               0.00 |        13.68 |                        0 |                       97.77 |                  99.23 |          98.64 |                98.70 |

|                                              |   max |·| (pp) |   signed value at max (pp) |   at (h) | state at max   |   at end (pp) |
|:---------------------------------------------|---------------:|---------------------------:|---------:|:---------------|--------------:|
| ('GDT 1 (LiNMC, 12 Jun)', 'OCV − CC')        |          0.995 |                      0.995 |    0.259 | charging       |         0.210 |
| ('GDT 1 (LiNMC, 12 Jun)', 'EKF − CC')        |          0.086 |                      0.086 |    0.986 | rest           |         0.086 |
| ('GDT 1 (LiNMC, 12 Jun)', 'EKF + KCL − CC')  |          0.063 |                      0.063 |    0.986 | rest           |         0.063 |
| ('GDT 1 (LiNMC, 12 Jun)', 'EKF + KCL − EKF') |          0.023 |                     -0.023 |    0.962 | rest           |        -0.023 |
| ('GDT 2 (LiNMC, 12 Jun)', 'OCV − CC')        |          2.632 |                      2.632 |    0.413 | charging       |         1.755 |
| ('GDT 2 (LiNMC, 12 Jun)', 'EKF − CC')        |          0.819 |                      0.819 |    0.991 | rest           |         0.819 |
| ('GDT 2 (LiNMC, 12 Jun)', 'EKF + KCL − CC')  |          0.791 |                      0.791 |    0.991 | rest           |         0.791 |
| ('GDT 2 (LiNMC, 12 Jun)', 'EKF + KCL − EKF') |          0.028 |                     -0.028 |    0.981 | rest           |        -0.028 |
| ('LiFePO4 (22 Jun)', 'OCV − CC')             |          5.729 |                     -5.729 |    0.160 | discharging    |        -5.292 |
| ('LiFePO4 (22 Jun)', 'EKF − CC')             |          5.384 |                     -5.384 |    0.725 | rest           |        -5.274 |
| ('LiFePO4 (22 Jun)', 'EKF + KCL − CC')       |          5.386 |                     -5.386 |    0.725 | rest           |        -5.275 |
| ('LiFePO4 (22 Jun)', 'EKF + KCL − EKF')      |          0.005 |                      0.005 |    0.374 | discharging    |        -0.000 |
| ('LiNMC (29 Jun)', 'OCV − CC')               |          5.716 |                      5.716 |    0.609 | charging       |         1.467 |
| ('LiNMC (29 Jun)', 'EKF − CC')               |          0.873 |                      0.873 |    1.655 | rest           |         0.873 |
| ('LiNMC (29 Jun)', 'EKF + KCL − CC')         |          0.933 |                      0.933 |    1.655 | rest           |         0.933 |
| ('LiNMC (29 Jun)', 'EKF + KCL − EKF')        |          0.059 |                      0.059 |    1.648 | rest           |         0.059 |

|                                          |   minutes |   OCV − CC |   EKF − CC |   EKF + KCL − CC |   EKF + KCL − EKF |
|:-----------------------------------------|----------:|-----------:|-----------:|-----------------:|------------------:|
| ('GDT 1 (LiNMC, 12 Jun)', 'charging')    |    50.600 |      3.919 |      0.082 |            0.061 |            -0.021 |
| ('GDT 1 (LiNMC, 12 Jun)', 'discharging') |     2.817 |     -2.093 |     -0.011 |           -0.011 |             0.001 |
| ('GDT 1 (LiNMC, 12 Jun)', 'rest')        |     5.733 |     -1.616 |      0.016 |            0.013 |            -0.003 |
| ('GDT 2 (LiNMC, 12 Jun)', 'charging')    |    55.567 |      3.329 |      0.780 |            0.754 |            -0.026 |
| ('GDT 2 (LiNMC, 12 Jun)', 'rest')        |     3.883 |     -1.574 |      0.039 |            0.037 |            -0.002 |
| ('LiFePO4 (22 Jun)', 'discharging')      |    18.550 |    -14.432 |     -7.064 |           -7.060 |             0.004 |
| ('LiFePO4 (22 Jun)', 'rest')             |    23.217 |      9.140 |      1.789 |            1.785 |            -0.004 |
| ('LiNMC (29 Jun)', 'charging')           |    53.483 |      4.651 |      0.770 |            0.810 |             0.040 |
| ('LiNMC (29 Jun)', 'discharging')        |    21.517 |     -0.860 |      0.033 |            0.050 |             0.017 |
| ('LiNMC (29 Jun)', 'pass-through')       |     0.183 |      0.008 |      0.004 |            0.003 |            -0.000 |
| ('LiNMC (29 Jun)', 'rest')               |    13.683 |     -2.332 |      0.068 |            0.070 |             0.002 |

In [ ]:
lims = {}
for run, o in OP.items():
    b = se.RUNS[run]['battery']
    v = 100 * o['est'][list(se.METHODS)].to_numpy()
    lo, hi = np.nanmin(v), np.nanmax(v)
    lims[b] = (min(lims.get(b, (lo, hi))[0], lo), max(lims.get(b, (lo, hi))[1], hi))
soc_ylim = {b: (np.floor(lo - 1), min(np.ceil(hi + 1), 100.5)) for b, (lo, hi) in lims.items()}
for i, (run, o) in enumerate(OP.items()):
    r = se.RUNS[run]
    title = f"{r['label']}: {r['battery']} in the {r['slot']} slot"
    fig = se.plot_operational_panels(o['win'], o['est'], o['state'], o['rests'], o['gaps'], title, r['slot'], soc_ylim[r['battery']])
    name = f"res_op_{run.replace(' ', '_')}_panels"
    se.save(fig, name)
    FIGS[{'LiNMC General Test': 'R5', 'LiFePO4 General Test': 'R6',
          'General Discharge Test': 'A3', 'General Discharge Test 2': 'A4'}[run]] = name
    plt.show()
fig = se.plot_pairwise_all({run: (o['est'], o['rests'], o['gaps']) for run, o in OP.items()})
se.save(fig, 'res_op_pairwise')
FIGS['R7'] = 'res_op_pairwise'
plt.show()

C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\3345338599.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\3345338599.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\3345338599.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\3345338599.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\3345338599.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.4 SoC behaviour during rest (Figures R8 and R9, Table R10)
- **Rest windows.** The same windows for every method (`find_rests`). Every current is below 0.5 A, and the rest is preceded by battery current above 1 A. t₀ is 10 s after the rest starts. The end is the last sample before any current exceeds 0.5 A.
- **Minimum length.** 5 min on the controlled runs, 2 min on the operational runs. A 1-min variant goes in the appendix.
- **Type.** Each rest is labelled by the sign of the battery current before it.
- **Controlled rows.** They come from the saved tables of `controlled_evaluation.ipynb`, and the LiNMC ones are re-checked below. On those runs, OCV mapping and the EKF use a model fitted to these same rests (in-sample).

In [11]:
longs = []
for b in ('LiNMC', 'LiFePO4'):
    d = pd.read_csv(se.TABLE_DIR / f'{b}_controlled_rest_deltas.csv')
    p = pd.read_csv(se.TABLE_DIR / f'{b}_controlled_rest_points.csv', index_col='rest')
    longs.append(se.rest_long(d, p, run=f'{b} controlled', battery=b, source='controlled'))
if CHECK_CONTROLLED:
    cfg = se.BATTERIES['LiNMC']
    win = W['LiNMC controlled']
    soc0 = se.characterization_soc(win, 'LiNMC', models['LiNMC'])[0]
    est_c, _ = se.run_methods(win, models['LiNMC'], se.settings('LiNMC', models['LiNMC'], 'tuned'), soc0=soc0)
    rests_c = se.find_rests(win)
    rests_c = rests_c[rests_c.session.isin(cfg['score_sessions'])]
    check = se.rest_deltas(est_c, rests_c).merge(longs[0], on=['rest', 'col'], suffixes=('', '_saved'))
    worst = (check.dsoc_t0_end_pp - check.dsoc_t0_end_pp_saved).abs().max()
    print(f'LiNMC controlled rest ΔSoC, recomputed against saved: largest difference {worst:.2e} pp')
    assert worst < 1e-6
REST_LIST = []
for run, o in OP.items():
    for key, rests in (('2 min', o['rests']), ('1 min', o['rests1'])):
        if not len(rests):
            continue
        p = se.rest_points(o['est'], rests)
        lab = dict(run=run, battery=se.RUNS[run]['battery'], source='operational', rule=key)
        part = se.rest_long(se.rest_deltas(o['est'], rests), p, **lab)
        inside = [o['gaps'][(o['gaps'].t > q.t_disc) & (o['gaps'].t <= q.t_end)].length_s.sum() for q in rests.itertuples()]
        part = part.join(pd.Series(inside, index=rests.rest.to_numpy(), name='unrecorded inside (s)'), on='rest')
        (longs if key == '2 min' else REST_LIST).append(part)
REST = pd.concat(longs, ignore_index=True)
REST1 = pd.concat(REST_LIST, ignore_index=True)
REST.to_csv(se.TABLE_DIR / 'res_rest_all.csv', index=False)
summary = se.rest_group_table(REST)
display(Markdown(se.md_table(summary, 'rest_summary')))
display(Markdown(se.md_table(se.rest_group_table(REST1), 'rest_summary_1min')))
rest_list = (REST.pivot_table(index=['source', 'run', 'rest'], columns='method', values='dsoc_t0_end_pp')[list(se.METHODS.values())]
             .join(REST.drop_duplicates(['run', 'rest']).set_index(['source', 'run', 'rest'])
                   [['type', 'length_min', 'criterion_met', 'dV_mV', 'unrecorded inside (s)']]))
display(Markdown(se.md_table(rest_list, 'rest_list', floatfmt='.4f')))

LiNMC controlled rest ΔSoC, recomputed against saved: largest difference 8.88e-16 pp


|                                                                   |   rests |   median |ΔSoC| (pp) |   max |ΔSoC| (pp) |   signed min (pp) |   signed max (pp) |   median length (min) |   median ΔV t₀→end (mV) |
|:------------------------------------------------------------------|--------:|---------------------:|------------------:|------------------:|------------------:|----------------------:|------------------------:|
| ('controlled', 'LiNMC', 'after_discharge', 'Coulomb counting')    |  13.000 |                0.000 |             0.000 |             0.000 |             0.000 |                31.279 |                 534.234 |
| ('controlled', 'LiNMC', 'after_discharge', 'OCV mapping')         |  13.000 |                4.579 |             8.145 |             1.683 |             8.145 |                31.279 |                 534.234 |
| ('controlled', 'LiNMC', 'after_discharge', 'EKF')                 |  13.000 |                0.005 |             0.068 |            -0.068 |             0.023 |                31.279 |                 534.234 |
| ('controlled', 'LiNMC', 'after_discharge', 'EKF + KCL')           |  13.000 |                0.010 |             0.059 |            -0.059 |             0.025 |                31.279 |                 534.234 |
| ('controlled', 'LiFePO4', 'after_discharge', 'Coulomb counting')  |   6.000 |                0.000 |             0.000 |             0.000 |             0.000 |                10.158 |                 499.813 |
| ('controlled', 'LiFePO4', 'after_discharge', 'OCV mapping')       |   6.000 |               23.087 |            37.558 |             4.056 |            37.558 |                10.158 |                 499.813 |
| ('controlled', 'LiFePO4', 'after_discharge', 'EKF')               |   6.000 |                0.000 |             0.019 |            -0.000 |             0.019 |                10.158 |                 499.813 |
| ('controlled', 'LiFePO4', 'after_discharge', 'EKF + KCL')         |   6.000 |                0.000 |             0.018 |             0.000 |             0.018 |                10.158 |                 499.813 |
| ('operational', 'LiNMC', 'after_charge', 'Coulomb counting')      |   1.000 |                0.000 |             0.000 |            -0.000 |            -0.000 |                 2.598 |                 -21.125 |
| ('operational', 'LiNMC', 'after_charge', 'OCV mapping')           |   1.000 |                0.182 |             0.182 |            -0.182 |            -0.182 |                 2.598 |                 -21.125 |
| ('operational', 'LiNMC', 'after_charge', 'EKF')                   |   1.000 |                0.024 |             0.024 |             0.024 |             0.024 |                 2.598 |                 -21.125 |
| ('operational', 'LiNMC', 'after_charge', 'EKF + KCL')             |   1.000 |                0.023 |             0.023 |             0.023 |             0.023 |                 2.598 |                 -21.125 |
| ('operational', 'LiFePO4', 'after_discharge', 'Coulomb counting') |   4.000 |                0.000 |             0.000 |             0.000 |             0.000 |                 3.315 |                  90.117 |
| ('operational', 'LiFePO4', 'after_discharge', 'OCV mapping')      |   4.000 |                0.424 |             1.022 |             0.278 |             1.022 |                 3.315 |                  90.117 |
| ('operational', 'LiFePO4', 'after_discharge', 'EKF')              |   4.000 |                0.343 |             0.635 |             0.217 |             0.635 |                 3.315 |                  90.117 |
| ('operational', 'LiFePO4', 'after_discharge', 'EKF + KCL')        |   4.000 |                0.342 |             0.636 |             0.214 |             0.636 |                 3.315 |                  90.117 |

|                                                                   |   rests |   median |ΔSoC| (pp) |   max |ΔSoC| (pp) |   signed min (pp) |   signed max (pp) |   median length (min) |   median ΔV t₀→end (mV) |
|:------------------------------------------------------------------|--------:|---------------------:|------------------:|------------------:|------------------:|----------------------:|------------------------:|
| ('operational', 'LiNMC', 'after_charge', 'Coulomb counting')      |   5.000 |                0.000 |             0.000 |            -0.000 |             0.000 |                 1.588 |                 -21.125 |
| ('operational', 'LiNMC', 'after_charge', 'OCV mapping')           |   5.000 |                0.146 |             0.205 |            -0.205 |            -0.029 |                 1.588 |                 -21.125 |
| ('operational', 'LiNMC', 'after_charge', 'EKF')                   |   5.000 |                0.009 |             0.029 |             0.003 |             0.029 |                 1.588 |                 -21.125 |
| ('operational', 'LiNMC', 'after_charge', 'EKF + KCL')             |   5.000 |                0.010 |             0.027 |             0.003 |             0.027 |                 1.588 |                 -21.125 |
| ('operational', 'LiNMC', 'after_discharge', 'Coulomb counting')   |   1.000 |                0.000 |             0.000 |             0.000 |             0.000 |                 1.723 |                  10.125 |
| ('operational', 'LiNMC', 'after_discharge', 'OCV mapping')        |   1.000 |                0.067 |             0.067 |             0.067 |             0.067 |                 1.723 |                  10.125 |
| ('operational', 'LiNMC', 'after_discharge', 'EKF')                |   1.000 |                0.000 |             0.000 |            -0.000 |            -0.000 |                 1.723 |                  10.125 |
| ('operational', 'LiNMC', 'after_discharge', 'EKF + KCL')          |   1.000 |                0.001 |             0.001 |            -0.001 |            -0.001 |                 1.723 |                  10.125 |
| ('operational', 'LiFePO4', 'after_discharge', 'Coulomb counting') |   4.000 |                0.000 |             0.000 |             0.000 |             0.000 |                 3.315 |                  90.117 |
| ('operational', 'LiFePO4', 'after_discharge', 'OCV mapping')      |   4.000 |                0.424 |             1.022 |             0.278 |             1.022 |                 3.315 |                  90.117 |
| ('operational', 'LiFePO4', 'after_discharge', 'EKF')              |   4.000 |                0.343 |             0.635 |             0.217 |             0.635 |                 3.315 |                  90.117 |
| ('operational', 'LiFePO4', 'after_discharge', 'EKF + KCL')        |   4.000 |                0.342 |             0.636 |             0.214 |             0.636 |                 3.315 |                  90.117 |

|                                                      |   Coulomb counting |   OCV mapping |     EKF |   EKF + KCL | type            |   length_min | criterion_met   |    dV_mV |   unrecorded inside (s) |
|:-----------------------------------------------------|-------------------:|--------------:|--------:|------------:|:----------------|-------------:|:----------------|---------:|------------------------:|
| ('controlled', 'LiFePO4 controlled', 1)              |             0.0000 |       33.9302 | -0.0000 |      0.0000 | after_discharge |       7.1274 | True            | 316.7344 |                nan      |
| ('controlled', 'LiFePO4 controlled', 2)              |             0.0000 |       37.5579 |  0.0000 |      0.0000 | after_discharge |       9.9698 | False           | 460.0313 |                nan      |
| ('controlled', 'LiFePO4 controlled', 3)              |             0.0000 |       27.3566 |  0.0000 |      0.0000 | after_discharge |      10.4204 | True            | 448.5937 |                nan      |
| ('controlled', 'LiFePO4 controlled', 4)              |             0.0000 |       18.8179 |  0.0000 |      0.0000 | after_discharge |      12.9508 | True            | 585.8750 |                nan      |
| ('controlled', 'LiFePO4 controlled', 5)              |             0.0000 |       11.1049 | -0.0000 |      0.0000 | after_discharge |      10.2301 | False           | 600.5625 |                nan      |
| ('controlled', 'LiFePO4 controlled', 6)              |             0.0000 |        4.0557 |  0.0190 |      0.0178 | after_discharge |      10.0866 | True            | 539.5938 |                nan      |
| ('controlled', 'LiNMC controlled', 1)                |             0.0000 |        4.6501 | -0.0176 |     -0.0176 | after_discharge |      16.9599 | True            | 458.3984 |                nan      |
| ('controlled', 'LiNMC controlled', 2)                |             0.0000 |        5.7619 | -0.0051 |     -0.0051 | after_discharge |      36.2722 | True            | 594.5312 |                nan      |
| ('controlled', 'LiNMC controlled', 3)                |             0.0000 |        4.3030 | -0.0252 |     -0.0236 | after_discharge |      32.8524 | True            | 727.6562 |                nan      |
| ('controlled', 'LiNMC controlled', 4)                |             0.0000 |        3.9905 |  0.0219 |      0.0223 | after_discharge |      31.5529 | True            | 550.0781 |                nan      |
| ('controlled', 'LiNMC controlled', 5)                |             0.0000 |        3.5130 |  0.0235 |      0.0249 | after_discharge |      20.0679 | True            | 396.0000 |                nan      |
| ('controlled', 'LiNMC controlled', 6)                |             0.0000 |        4.5793 |  0.0122 |      0.0131 | after_discharge |      16.7879 | True            | 399.1875 |                nan      |
| ('controlled', 'LiNMC controlled', 7)                |             0.0000 |        7.4524 |  0.0038 |      0.0044 | after_discharge |      21.1278 | True            | 448.8281 |                nan      |
| ('controlled', 'LiNMC controlled', 8)                |             0.0000 |        8.1449 |  0.0029 |      0.0032 | after_discharge |      31.2793 | True            | 534.2344 |                nan      |
| ('controlled', 'LiNMC controlled', 9)                |             0.0000 |        7.5550 |  0.0052 |      0.0053 | after_discharge |      28.1488 | True            | 577.0000 |                nan      |
| ('controlled', 'LiNMC controlled', 10)               |             0.0000 |        4.3738 |  0.0055 |      0.0076 | after_discharge |      33.3601 | True            | 541.5625 |                nan      |
| ('controlled', 'LiNMC controlled', 11)               |             0.0000 |        7.4160 |  0.0024 |      0.0036 | after_discharge |      33.4961 | True            | 377.8906 |                nan      |
| ('controlled', 'LiNMC controlled', 12)               |             0.0000 |        2.4038 |  0.0037 |      0.0103 | after_discharge |      29.2901 | True            | 519.8437 |                nan      |
| ('controlled', 'LiNMC controlled', 13)               |             0.0000 |        1.6832 | -0.0678 |     -0.0591 | after_discharge |      31.3383 | True            | 833.8125 |                nan      |
| ('operational', 'General Discharge Test 2', 1)       |            -0.0000 |       -0.1815 |  0.0242 |      0.0231 | after_charge    |       2.5981 | True            | -21.1250 |                  0.0000 |
| ('operational', 'LiFePO4 Battery Discharge Test', 1) |             0.0000 |        1.0216 |  0.6348 |      0.6364 | after_discharge |       3.5648 | False           | 208.1250 |                  0.0000 |
| ('operational', 'LiFePO4 Battery Discharge Test', 2) |             0.0000 |        0.3380 |  0.2316 |      0.2312 | after_discharge |       3.0652 | False           |  69.6094 |                  0.0000 |
| ('operational', 'LiFePO4 Battery Discharge Test', 3) |             0.0000 |        0.2785 |  0.2172 |      0.2143 | after_discharge |       2.1448 | False           |  59.6250 |                 10.9991 |
| ('operational', 'LiFePO4 Battery Discharge Test', 4) |             0.0000 |        0.5110 |  0.4540 |      0.4530 | after_discharge |      11.2452 | True            | 110.6250 |                124.9991 |

In [12]:
groups = {}
for key, label in ((('operational', 'LiNMC', 'after_charge'), 'Operational, LiNMC\nafter charge'),
                   (('operational', 'LiFePO4', 'after_discharge'), 'Operational, LiFePO4\nafter discharge'),
                   (('operational', 'LiNMC', 'after_discharge'), 'Operational, LiNMC\nafter discharge'),
                   (('operational', 'LiFePO4', 'after_charge'), 'Operational, LiFePO4\nafter charge'),
                   (('controlled', 'LiNMC', 'after_discharge'), 'Controlled, LiNMC\nafter discharge (in-sample)'),
                   (('controlled', 'LiFePO4', 'after_discharge'), 'Controlled, LiFePO4\nafter discharge (in-sample)')):
    if ((REST.source == key[0]) & (REST.battery == key[1]) & (REST.type == key[2])).any():
        groups[key] = label
fig = se.plot_rest_pairs(REST, groups)
se.save(fig, 'res_rest_pairs')
FIGS['R9'] = 'res_rest_pairs'
plt.show()

cases = []
for typ, label in (('after_discharge', 'after discharge'), ('after_charge', 'after charge')):
    cand = [(run, q) for run, o in OP.items() for q in o['rests'].itertuples() if q.type == typ]
    if not cand:
        continue
    lengths = np.array([q.t_end - q.t_disc for _, q in cand])
    run, q = cand[int(np.argmin(np.abs(lengths - np.median(lengths))))]       # the median-length rest of this type
    cases.append((OP[run]['est'], q, f"{se.RUNS[run]['label']}, rest {q.rest}, {label}, {(q.t_end - q.t_disc) / 60:.1f} min"))
    NUM[f'close-up {label}'] = f"{run}, rest {q.rest}"
fig = se.plot_rest_closeups(cases)
se.save(fig, 'res_rest_closeups')
FIGS['R8'] = 'res_rest_closeups'
plt.show()

C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\1899181193.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\1899181193.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.5 Sensitivity to the initial SoC (Figure R10, Table R11)
- **Replays.** Each operational run is replayed with frozen settings from the fixed starts in `se.INIT_GRID` (10, 30, 50, 70 and 90 %), the same for every run.
- **Spread.** Measured within each method: max − min across its replays at each record.
- **Threshold.** 1 pp, set before the replays were run. "Below from (h) to the end" is the time after which the spread never returns above it.
- OCV mapping has no initial state, so it is marked not applicable.

In [13]:
init = pd.concat({se.RUNS[run]['short']: se.init_spread_table(o['est'], o['extra'], o['gaps']) for run, o in OP.items()},
                 names=['run', 'method'])
display(Markdown(se.md_table(init, 'init', floatfmt='.4g')))
fig = se.plot_init({run: (o['est'], o['extra'], o['gaps']) for run, o in OP.items()})
se.save(fig, 'res_init_spread')
FIGS['R10'] = 'res_init_spread'
plt.show()
for run, o in OP.items():
    k_end = np.flatnonzero(o['est'].V > se.V_CUTOFF)[-1]
    NUM[f'{run}: EKF end SoC across starts (%)'] = np.round(100 * o['extra']['by_start']['soc_ekf'][1:, k_end], 3).tolist()

|                                               | initial values (%)   | spread at the first record (pp)   | spread at the end (pp)   | first below 1 pp (h)   | below 1 pp from (h) to the end   |
|:----------------------------------------------|:---------------------|:----------------------------------|:-------------------------|:-----------------------|:---------------------------------|
| ('GDT 1 (LiNMC, 12 Jun)', 'Coulomb counting') | 10, 30, 50, 70, 90   | 80                                | 80                       | not reached            | not reached                      |
| ('GDT 1 (LiNMC, 12 Jun)', 'EKF')              | 10, 30, 50, 70, 90   | 10.65                             | 0.0006339                | 0.00028                | 0.00028                          |
| ('GDT 1 (LiNMC, 12 Jun)', 'EKF + KCL')        | 10, 30, 50, 70, 90   | 10.65                             | 0.0006153                | 0.00028                | 0.00028                          |
| ('GDT 1 (LiNMC, 12 Jun)', 'OCV mapping')      | not applicable       | not applicable                    | not applicable           | not applicable         | not applicable                   |
| ('GDT 2 (LiNMC, 12 Jun)', 'Coulomb counting') | 10, 30, 50, 70, 90   | 80                                | 80                       | not reached            | not reached                      |
| ('GDT 2 (LiNMC, 12 Jun)', 'EKF')              | 10, 30, 50, 70, 90   | 19.51                             | 0.007532                 | 0.0002799              | 0.0002799                        |
| ('GDT 2 (LiNMC, 12 Jun)', 'EKF + KCL')        | 10, 30, 50, 70, 90   | 19.51                             | 0.007582                 | 0.0002799              | 0.0002799                        |
| ('GDT 2 (LiNMC, 12 Jun)', 'OCV mapping')      | not applicable       | not applicable                    | not applicable           | not applicable         | not applicable                   |
| ('LiFePO4 (22 Jun)', 'Coulomb counting')      | 10, 30, 50, 70, 90   | 80                                | 80                       | not reached            | not reached                      |
| ('LiFePO4 (22 Jun)', 'EKF')                   | 10, 30, 50, 70, 90   | 79.62                             | 0.0001011                | 0.3425                 | 0.3425                           |
| ('LiFePO4 (22 Jun)', 'EKF + KCL')             | 10, 30, 50, 70, 90   | 79.62                             | 0.0001013                | 0.3425                 | 0.3425                           |
| ('LiFePO4 (22 Jun)', 'OCV mapping')           | not applicable       | not applicable                    | not applicable           | not applicable         | not applicable                   |
| ('LiNMC (29 Jun)', 'Coulomb counting')        | 10, 30, 50, 70, 90   | 80                                | 80                       | not reached            | not reached                      |
| ('LiNMC (29 Jun)', 'EKF')                     | 10, 30, 50, 70, 90   | 14.05                             | 0.0001842                | 0.00028                | 0.00028                          |
| ('LiNMC (29 Jun)', 'EKF + KCL')               | 10, 30, 50, 70, 90   | 14.05                             | 0.0002031                | 0.00028                | 0.00028                          |
| ('LiNMC (29 Jun)', 'OCV mapping')             | not applicable       | not applicable                    | not applicable           | not applicable         | not applicable                   |

C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\2197753918.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4.6 Effect of the KCL correction (Figure R11, Tables R12–R14)
**Residual.** r = I_load − I_MPPT − I_bat,main − I_bat,alternate (A).
- Sign convention: battery current is positive on discharge, MPPT current positive out of the MPPT, load current positive into the load.
- A slot whose voltage is below 30 V is disconnected, and its current is set to 0 A, as the server does.
- Measured branches: MPPT net output (HE1), load and motor controller (HE4), main battery (HE2), alternate battery (HE3).

**What is compared.** EKF and EKF + KCL share every setting. They differ only in the current they integrate: offsets only, or offsets plus the gated KCL update.

**Exclusions.** Records within 10 s after an inferred state change, or after an unrecorded interval, are left out of the per-state table.

In [ ]:
kcl = pd.concat({se.RUNS[run]['short']: se.kcl_state_table(o['est'], o['extra'], o['state'], o['gaps']) for run, o in OP.items()},
                names=['run', 'state'])
display(Markdown(se.md_table(kcl, 'kcl_by_state', floatfmt='.4f')))
ah = pd.DataFrame({se.RUNS[run]['short']: {'raw (Ah)': o['extra']['Ah']['raw'], 'offsets only (Ah)': o['extra']['Ah']['offsets'],
                                           'offsets + gated KCL (Ah)': o['extra']['Ah']['kcl'],
                                           'gated KCL − offsets only (Ah)': o['extra']['Ah']['kcl'] - o['extra']['Ah']['offsets']}
                   for run, o in OP.items()}).T
display(Markdown(se.md_table(ah, 'kcl_charge')))
for i, (run, o) in enumerate(OP.items()):
    r = se.RUNS[run]
    fig = se.plot_kcl_channels(o['win'], o['est'], o['extra'], o['state'], o['gaps'],
                               f"{r['label']}: current-balance residual, offsets removed and EKF + KCL minus EKF", r['slot'])
    name = f"res_kcl_{run.replace(' ', '_')}"
    se.save(fig, name)
    FIGS[{'LiNMC General Test': 'R11', 'LiFePO4 General Test': 'A5',
          'General Discharge Test': 'A6', 'General Discharge Test 2': 'A7'}[run]] = name
    plt.show()

# The 29 Jun run as the server processed it (MPPT flip latch on): saved by operational_evaluation.ipynb, section 2.8.
# The "sign as recorded" column must match this notebook's own run.
latch = pd.read_csv(se.TABLE_DIR / 'op_LiNMC_Battery_Discharge_Test_latch_evidence.csv', index_col=0)
x = OP['LiNMC General Test']['extra']
assert abs(latch.iloc[0, 0] - x['Ah']['offsets']) < 2e-3 and abs(latch.iloc[1, 0] - x['Ah']['kcl']) < 2e-3
display(Markdown(se.md_table(latch, 'kcl_latch')))

|                                          |   minutes |   median |r| raw (A) |   median |r| after offsets (A) |   median |r| after gated KCL (A) |   median |EKF+KCL − EKF| (pp) |   max |EKF+KCL − EKF| (pp) |   net change of EKF+KCL − EKF in state (pp) |
|:-----------------------------------------|----------:|---------------------:|-------------------------------:|---------------------------------:|------------------------------:|---------------------------:|--------------------------------------------:|
| ('GDT 1 (LiNMC, 12 Jun)', 'charging')    |   49.7168 |               0.1591 |                         0.0548 |                           0.0312 |                        0.0148 |                     0.0230 |                                     -0.0211 |
| ('GDT 1 (LiNMC, 12 Jun)', 'discharging') |    2.1001 |               0.1827 |                         0.0299 |                           0.0331 |                        0.0003 |                     0.0012 |                                      0.0002 |
| ('GDT 1 (LiNMC, 12 Jun)', 'rest')        |    4.5833 |               0.1920 |                         0.0344 |                           0.0337 |                        0.0010 |                     0.0231 |                                     -0.0018 |
| ('GDT 1 (LiNMC, 12 Jun)', 'all states')  |   56.4002 |               0.1619 |                         0.0523 |                           0.0315 |                        0.0137 |                     0.0231 |                                     -0.0226 |
| ('GDT 2 (LiNMC, 12 Jun)', 'charging')    |   55.0667 |               0.0804 |                         0.1455 |                           0.0290 |                        0.0154 |                     0.0260 |                                     -0.0262 |
| ('GDT 2 (LiNMC, 12 Jun)', 'rest')        |    3.2168 |               0.1060 |                         0.0345 |                           0.0271 |                        0.0274 |                     0.0275 |                                     -0.0013 |
| ('GDT 2 (LiNMC, 12 Jun)', 'all states')  |   58.2835 |               0.0820 |                         0.1420 |                           0.0289 |                        0.0160 |                     0.0275 |                                     -0.0274 |
| ('LiFePO4 (22 Jun)', 'discharging')      |   17.2167 |               0.0310 |                         0.0307 |                           0.0287 |                        0.0016 |                     0.0052 |                                      0.0014 |
| ('LiFePO4 (22 Jun)', 'rest')             |   21.0501 |               0.0200 |                         0.0202 |                           0.0211 |                        0.0001 |                     0.0032 |                                     -0.0028 |
| ('LiFePO4 (22 Jun)', 'all states')       |   38.2668 |               0.0240 |                         0.0238 |                           0.0238 |                        0.0004 |                     0.0052 |                                     -0.0014 |
| ('LiNMC (29 Jun)', 'charging')           |   51.4333 |               0.2750 |                         0.2199 |                           0.0484 |                        0.0147 |                     0.0578 |                                      0.0394 |
| ('LiNMC (29 Jun)', 'discharging')        |   19.2502 |               0.1077 |                         0.1131 |                           0.0406 |                        0.0042 |                     0.0222 |                                      0.0157 |
| ('LiNMC (29 Jun)', 'rest')               |   10.4169 |               0.0294 |                         0.0400 |                           0.0564 |                        0.0000 |                     0.0593 |                                      0.0003 |
| ('LiNMC (29 Jun)', 'all states')         |   81.1003 |               0.2402 |                         0.1810 |                           0.0468 |                        0.0098 |                     0.0593 |                                      0.0554 |

|                       |   raw (Ah) |   offsets only (Ah) |   offsets + gated KCL (Ah) |   gated KCL − offsets only (Ah) |
|:----------------------|-----------:|--------------------:|---------------------------:|--------------------------------:|
| GDT 1 (LiNMC, 12 Jun) |     -1.611 |              -1.512 |                     -1.469 |                           0.043 |
| GDT 2 (LiNMC, 12 Jun) |     -1.841 |              -1.787 |                     -1.747 |                           0.040 |
| LiFePO4 (22 Jun)      |      0.585 |               0.584 |                      0.585 |                           0.001 |
| LiNMC (29 Jun)        |     -2.853 |              -3.106 |                     -3.177 |                          -0.071 |

C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\592720772.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\592720772.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\592720772.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Kor\AppData\Local\Temp\ipykernel_35320\592720772.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


|                                           |   MPPT sign as recorded |   as the server processed it (flip latch on) |
|:------------------------------------------|------------------------:|---------------------------------------------:|
| battery charge out, offsets (Ah)          |                  -3.106 |                                       -3.106 |
| battery charge out, gated KCL (Ah)        |                  -3.177 |                                       -2.456 |
| largest |battery bias|, gated KCL (A)     |                   0.488 |                                        2.617 |
| largest |MPPT bias|, gated KCL (A)        |                   0.095 |                                        0.610 |
| EKF + KCL − EKF at end (pp)               |                   0.059 |                                       -0.508 |
| largest per-minute |EKF + KCL − EKF| (pp) |                   0.059 |                                        0.514 |
| voltage innovation RMS, EKF (mV)          |                  15.025 |                                       15.025 |
| voltage innovation RMS, EKF + KCL (mV)    |                  14.482 |                                       22.968 |
| minutes with |KCL residual| > 2 A         |                   0.000 |                                       18.000 |

## Numbers quoted in the text, figure map, and copy to `results_section/`

In [15]:
for k, v in NUM.items():
    print(f'{k}: {v}')

shared settings: All methods: 10-sample median stream (as kalman_filter.js), samples below the 30 V cutoff excluded, records every 1 s. Offset correction (shared input step, not a method): a sensor is idle when load or MPPT reads < 0.3 A, or a battery reads < 1.0 A while load and MPPT are idle, for 10 s; its offset is learned while idle (R 0.001 A²·s ÷ dt, process noise 1e-07 A²/s) and subtracted while active; readings below 0.3 A are output as 0 A.
LiNMC: largest |log-model residual|: {'soc': 17.22, 'V': 49.44, 'log_mV': -302.22}
LiNMC: largest |leave-one-out residual|: {'soc': 10.17, 'V': 49.1, 'loo_mV': 727.73}
LiNMC: rests where 10 mV/min was not reached (SoC %): []
LiFePO4: largest |log-model residual|: {'soc': 34.83, 'V': 52.19, 'log_mV': 63.15}
LiFePO4: largest |leave-one-out residual|: {'soc': 10.96, 'V': 51.24, 'loo_mV': 344.67}
LiFePO4: rests where 10 mV/min was not reached (SoC %): [59.3, 22.8]
close-up after discharge: LiFePO4 Battery Discharge Test, rest 1
close-up after c

In [ ]:
SOURCES = {
    'R1': ('4.1', 'LiNMC Controlled Discharge', 'load_run, window_median, find_rests, active_blocks, plot_char_run'),
    'R2': ('4.1', 'LiFePO4 Controlled Discharge', 'load_run, window_median, find_rests, active_blocks, plot_char_run'),
    'R3': ('4.1', '3.ocv-soc-fit.ipynb outputs (both batteries), points_of_interest.json', 'ocv_knots, ocv_fit_residuals, plot_rest_voltage_fit'),
    'R4': ('4.2', 'all six runs', 'idle_windows, learned_offsets (offsets from kcl_batch or run_methods with OFFSETS), plot_offset_timeline'),
    'R5': ('4.3', 'LiNMC General Test', 'run_operational, record_state, find_rests, gap_table, plot_operational_panels'),
    'R6': ('4.3', 'LiFePO4 General Test', 'run_operational, record_state, find_rests, gap_table, plot_operational_panels'),
    'R7': ('4.3', 'four operational runs', 'pairwise, plot_pairwise_all'),
    'R8': ('4.4', 'operational runs (median-length rest of each type)', 'find_rests, plot_rest_closeups'),
    'R9': ('4.4', 'operational runs + controlled rest tables', 'rest_deltas, rest_points, rest_long, plot_rest_pairs'),
    'R10': ('4.5', 'four operational runs', 'run_operational (INIT_GRID replays), plot_init'),
    'R11': ('4.6', 'LiNMC General Test', 'run_methods (OFFSETS, KCL_GATED), plot_kcl_channels'),
    'A3': ('4.3', 'General Discharge Test', 'plot_operational_panels'),
    'A4': ('4.3', 'General Discharge Test 2', 'plot_operational_panels'),
    'A5': ('4.6', 'LiFePO4 General Test', 'plot_kcl_channels'),
    'A6': ('4.6', 'General Discharge Test', 'plot_kcl_channels'),
    'A7': ('4.6', 'General Discharge Test 2', 'plot_kcl_channels'),
}
# Supplementary figures made by the other notebooks (copied as they are)
EXISTING = {'A1': ('LiNMC_controlled_whole_run', 'controlled_evaluation.ipynb (BATTERY = LiNMC)'),
            'A2': ('LiFePO4_controlled_whole_run', 'controlled_evaluation.ipynb (BATTERY = LiFePO4)'),
            'A8': ('LiNMC_controlled_rest_summary', 'controlled_evaluation.ipynb (BATTERY = LiNMC)'),
            'A9': ('LiFePO4_controlled_rest_summary', 'controlled_evaluation.ipynb (BATTERY = LiFePO4)'),
            'A10': ('LiNMC_controlled_init_spread', 'controlled_evaluation.ipynb (BATTERY = LiNMC)'),
            'A11': ('LiFePO4_controlled_init_spread', 'controlled_evaluation.ipynb (BATTERY = LiFePO4)'),
            'A12': ('op_LiNMC_Battery_Discharge_Test_kcl_server_latch', 'operational_evaluation.ipynb, section 2.8')}
rows = [{'figure': k, 'file': f'figures/{v}.png', 'made by': f'results_chapter.ipynb, section {SOURCES[k][0]}',
         'data': SOURCES[k][1], 'soc_eval functions': SOURCES[k][2]} for k, v in FIGS.items()]
rows += [{'figure': k, 'file': f'figures/{v}.png', 'made by': nb, 'data': 'see that notebook', 'soc_eval functions': '–'}
         for k, (v, nb) in EXISTING.items()]
fmap = pd.DataFrame(rows).sort_values('figure', key=lambda s: s.map(lambda x: (x[0], int(x[1:])))).reset_index(drop=True)
display(Markdown(se.md_table(fmap, 'figure_map', index=False)))

out = se.RESULTS_DIR / 'figures'
out.mkdir(parents=True, exist_ok=True)
for name in list(FIGS.values()) + [v for v, _ in EXISTING.values()]:
    shutil.copy2(se.FIG_DIR / f'{name}.png', out / f'{name}.png')
print(f'Copied {len(FIGS) + len(EXISTING)} figures to {out}')

| figure   | file                                                         | made by                                         | data                                                                        | soc_eval functions                                                                                       |
|:---------|:-------------------------------------------------------------|:------------------------------------------------|:----------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------|
| A1       | figures/LiNMC_controlled_whole_run.png                       | controlled_evaluation.ipynb (BATTERY = LiNMC)   | see that notebook                                                           | –                                                                                                        |
| A2       | figures/LiFePO4_controlled_whole_run.png                     | controlled_evaluation.ipynb (BATTERY = LiFePO4) | see that notebook                                                           | –                                                                                                        |
| A3       | figures/res_op_General_Discharge_Test_panels.png             | results_chapter.ipynb, section 4.3              | General Discharge Test                                                      | plot_operational_panels                                                                                  |
| A4       | figures/res_op_General_Discharge_Test_2_panels.png           | results_chapter.ipynb, section 4.3              | General Discharge Test 2                                                    | plot_operational_panels                                                                                  |
| A5       | figures/res_kcl_LiFePO4_Battery_Discharge_Test.png           | results_chapter.ipynb, section 4.6              | LiFePO4 Battery Discharge Test                                              | plot_kcl_channels                                                                                        |
| A6       | figures/res_kcl_General_Discharge_Test.png                   | results_chapter.ipynb, section 4.6              | General Discharge Test                                                      | plot_kcl_channels                                                                                        |
| A7       | figures/res_kcl_General_Discharge_Test_2.png                 | results_chapter.ipynb, section 4.6              | General Discharge Test 2                                                    | plot_kcl_channels                                                                                        |
| A8       | figures/LiNMC_controlled_rest_summary.png                    | controlled_evaluation.ipynb (BATTERY = LiNMC)   | see that notebook                                                           | –                                                                                                        |
| A9       | figures/LiFePO4_controlled_rest_summary.png                  | controlled_evaluation.ipynb (BATTERY = LiFePO4) | see that notebook                                                           | –                                                                                                        |
| A10      | figures/LiNMC_controlled_init_spread.png                     | controlled_evaluation.ipynb (BATTERY = LiNMC)   | see that notebook                                                           | –                                                                                                        |
| A11      | figures/LiFePO4_controlled_init_spread.png                   | controlled_evaluation.ipynb (BATTERY = LiFePO4) | see that notebook                                                           | –                                                                                                        |
| A12      | figures/op_LiNMC_Battery_Discharge_Test_kcl_server_latch.png | operational_evaluation.ipynb, section 2.8       | see that notebook                                                           | –                                                                                                        |
| R1       | figures/res_char_run_LiNMC.png                               | results_chapter.ipynb, section 4.1              | LiNMC Remapped                                                              | load_run, window_median, find_rests, active_blocks, plot_char_run                                        |
| R2       | figures/res_char_run_LiFePO4.png                             | results_chapter.ipynb, section 4.1              | LiFePO4 Remapped                                                            | load_run, window_median, find_rests, active_blocks, plot_char_run                                        |
| R3       | figures/res_rest_voltage_fit.png                             | results_chapter.ipynb, section 4.1              | 3.ocv-soc-fit.ipynb outputs (both batteries), points_of_interest.json | ocv_knots, ocv_fit_residuals, plot_rest_voltage_fit                                                      |
| R4       | figures/res_offset_timeline.png                              | results_chapter.ipynb, section 4.2              | all six runs                                                                | idle_windows, learned_offsets (offsets from kcl_batch or run_methods with OFFSETS), plot_offset_timeline |
| R5       | figures/res_op_LiNMC_Battery_Discharge_Test_panels.png       | results_chapter.ipynb, section 4.3              | LiNMC Battery Discharge Test                                                | run_operational, record_state, find_rests, gap_table, plot_operational_panels                            |
| R6       | figures/res_op_LiFePO4_Battery_Discharge_Test_panels.png     | results_chapter.ipynb, section 4.3              | LiFePO4 Battery Discharge Test                                              | run_operational, record_state, find_rests, gap_table, plot_operational_panels                            |
| R7       | figures/res_op_pairwise.png                                  | results_chapter.ipynb, section 4.3              | four operational runs                                                       | pairwise, plot_pairwise_all                                                                              |
| R8       | figures/res_rest_closeups.png                                | results_chapter.ipynb, section 4.4              | operational runs (median-length rest of each type)                          | find_rests, plot_rest_closeups                                                                           |
| R9       | figures/res_rest_pairs.png                                   | results_chapter.ipynb, section 4.4              | operational runs + controlled rest tables                                   | rest_deltas, rest_points, rest_long, plot_rest_pairs                                                     |
| R10      | figures/res_init_spread.png                                  | results_chapter.ipynb, section 4.5              | four operational runs                                                       | run_operational (INIT_GRID replays), plot_init                                                           |
| R11      | figures/res_kcl_LiNMC_Battery_Discharge_Test.png             | results_chapter.ipynb, section 4.6              | LiNMC Battery Discharge Test                                                | run_methods (OFFSETS, KCL_GATED), plot_kcl_channels                                                      |

Copied 23 figures to C:\Users\Kor\Documents\GitHub\solar_proa_network_server\final_report\results_section\figures
